# 第三期只读数据核查

来源：项目 `.env.local` 指向的测试副本。每条连接强制只读，复用质量接口的 Mapper SQL，不输出凭据、人员或原始附件地址。

2026-10-26 已执行审计的原始行基线：巡检 2,583,366，病害 14,382，维修 11,888；可归属当前有效桥梁分别为 2,524,426、12,460、11,664。完整证据见 `../runtime/phase-three-audit.json` 和 `phase-three-audit.md`。

此 Notebook 的代码单元留待显式运行；其审计函数已通过脚本执行。运行将重新查询当前副本，日期和计数可能随来源更新变化。


In [ ]:
import importlib.util
import json
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "scripts").exists():
    root = root.parent
assert (root / "scripts/audit-phase-three.py").exists(), "请从项目根目录或 docs 启动 Notebook"
sys.path.insert(0, str(root / "scripts"))
spec = importlib.util.spec_from_file_location("phase_three_audit", root / "scripts/audit-phase-three.py")
audit_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit_module)


## 重跑审计

口径 RAW_IMPORT 包含原始重复、删除及未归属行。记录标识粒度为 `(sscsdm,bsm)`。单桥 API 另按最新有效版本查询；当前副本没有重复或删除行，两种数量才可对照。


In [ ]:
data = audit_module.audit()
profiles = {kind: table["profile"] for kind, table in data["tables"].items()}
print(json.dumps(profiles, ensure_ascii=False, indent=2, default=str))


## 检查可持续维护的约束

不把未关联数据视为不存在，不把空巡检结果视为正常，不要求业务病害编号唯一；仅记录真实问题并核查关联口径。


In [ ]:
for kind, profile in profiles.items():
    assert profile["raw_rows"] == profile["mapped_rows"] + profile["unmatched_rows"] + profile["ambiguous_rows"]
    assert profile["raw_rows"] == profile["distinct_keys"] + profile["duplicate_excess"] + profile["missing_identifiers"]
relations = data["currentMappedRelations"]
print(json.dumps(relations, ensure_ascii=False, indent=2))
print("补充养护表原始行/可关联行:", data["supplementaryMaintenanceRows"], data["supplementaryMaintenanceMappedRows"])


## 查看实际查询和字段结构

渲染器只支持已知 Mapper 的 include、条件分支与参数绑定，不执行 XML 中的任意代码。以下查询使用参数化执行；SQL 文件中 `%` 是 psycopg 转义形式。


In [ ]:
query, parameters = audit_module.mapper_query(
    "com.bridgemind.work.mapper.QualityMapper.profile", {"kind": "defects", "city": "320100"}
)
print(query)
print("参数:", parameters)
print(json.dumps({kind: table["columns"] for kind, table in data["tables"].items()}, ensure_ascii=False, indent=2))
